<a href="https://colab.research.google.com/github/daniellelooo/calidad-mineria-datos/blob/main/03_despliegue_gui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 3 — Despliegue con Interfaz Gráfica
## Práctica 4: Calidad y Minería de Datos en Python

**Objetivo:** desplegar el modelo predictivo entrenado y hiperparametrizado en el Notebook 2 a través de una **interfaz gráfica con Tkinter** que permita al usuario ingresar los datos de un deportista y obtener la predicción del Sector (OLIMPICO / PARALIMPICO).

**Insumos:**
- `modelo_final.joblib` (generado por el Notebook 2): contiene el modelo, el escalador, el codificador del target y la lista de features.

**Tecnología:** Tkinter (librería estándar de Python — no requiere instalación adicional).

## 1. Carga de artefactos del modelo

In [1]:
import joblib
import numpy as np
import pandas as pd

artefactos = joblib.load('modelo_final.joblib')

modelo            = artefactos['modelo']
scaler            = artefactos['scaler']
label_encoder     = artefactos['label_encoder']
feature_columns   = artefactos['feature_columns']
requires_scaling  = artefactos['requires_scaling']
modelo_nombre     = artefactos['modelo_nombre']
best_params       = artefactos['best_params']

print(f"Modelo cargado: {modelo_nombre}")
print(f"Escalado requerido: {requires_scaling}")
print(f"Features esperadas ({len(feature_columns)}):")
for c in feature_columns:
    print(f"  - {c}")
print(f"\nClases del modelo: {list(label_encoder.classes_)}")
print(f"\nMejores hiperparametros:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

Modelo cargado: Arbol de Decision
Escalado requerido: False
Features esperadas (7):
  - VIGENCIA
  - Inversion_por_deportista
  - Grupo_Etario_Adolescencia_12_18
  - Grupo_Etario_Adultez_27_59
  - Grupo_Etario_Infancia_6_11
  - Grupo_Etario_Juventud_19_26
  - Grupo_Etario_Primera_Infancia_0_5

Clases del modelo: ['OLIMPICO', 'PARALIMPICO']

Mejores hiperparametros:
  criterion: gini
  max_depth: None
  min_samples_leaf: 1
  min_samples_split: 2


## 2. Función de predicción

Crea un vector de features en el orden correcto a partir de los inputs del usuario, aplica escalado si el modelo lo requiere y devuelve la clase predicha junto con la probabilidad.

In [2]:
def predecir(vigencia: int, grupo_etario: str, inversion: float):
    """Construye el vector de features y devuelve (clase, probabilidad_paralimpico)."""

    # Inicializar fila con ceros
    fila = pd.DataFrame([{c: 0 for c in feature_columns}])

    # Asignar valores numericos
    fila['VIGENCIA'] = vigencia
    fila['Inversion_por_deportista'] = inversion

    # Asignar one-hot del grupo etario
    columna_grupo = f"Grupo_Etario_{grupo_etario.replace(' ', '_').replace('-', '_')}"
    if columna_grupo in fila.columns:
        fila[columna_grupo] = 1
    else:
        raise ValueError(f"Grupo etario no reconocido: {grupo_etario}")

    # Reordenar columnas en el orden exacto que espera el modelo
    fila = fila[feature_columns]

    # Escalar si corresponde
    X = scaler.transform(fila) if requires_scaling else fila

    # Predecir
    clase_idx = modelo.predict(X)[0]
    clase = label_encoder.inverse_transform([clase_idx])[0]
    proba_paralimpico = modelo.predict_proba(X)[0][int(label_encoder.transform(['PARALIMPICO'])[0])]

    return clase, float(proba_paralimpico)


# Prueba rapida
clase, p = predecir(2024, 'Adolescencia 12-18', 3310519.10)
print(f"Prueba: clase={clase}, P(PARALIMPICO)={p:.4f}")

Prueba: clase=OLIMPICO, P(PARALIMPICO)=0.4694


## 3. Interfaz gráfica con Tkinter

La interfaz tiene los siguientes campos:
- **Vigencia (año):** dropdown con 2022, 2023, 2024
- **Grupo etario:** dropdown con las 5 categorías
- **Inversión por deportista (COP):** campo numérico
- **Botón Predecir** que muestra el resultado y la probabilidad
- **Botón Limpiar** para reiniciar los campos

Para ejecutar la GUI corre la siguiente celda en un entorno con display gráfico (Jupyter local, escritorio, etc.).

> **Nota:** En entornos sin display (servidores remotos, Colab, contenedores) Tkinter mostrará error. En esos casos usa el script alternativo `app_tkinter.py` que se exporta en la celda 4.

In [3]:
import tkinter as tk
from tkinter import ttk, messagebox

GRUPOS_ETARIOS = [
    'Primera Infancia 0-5',
    'Infancia 6-11',
    'Adolescencia 12-18',
    'Juventud 19-26',
    'Adultez 27-59'
]

VIGENCIAS = [2022, 2023, 2024]


class PredictorApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Predictor Sector - Escuela de Talentos")
        self.root.geometry("520x520")
        self.root.configure(bg='#F5F5F5')

        # Estilo
        style = ttk.Style()
        style.theme_use('clam')
        style.configure('TLabel', background='#F5F5F5', font=('Segoe UI', 10))
        style.configure('TButton', font=('Segoe UI', 10, 'bold'))
        style.configure('Header.TLabel', font=('Segoe UI', 14, 'bold'),
                       background='#2E75B6', foreground='white')

        # Header
        header = ttk.Label(root, text="  Predictor de Sector Deportivo",
                          style='Header.TLabel', anchor='w')
        header.pack(fill='x', ipady=12)

        subt = ttk.Label(root,
                         text=f"Modelo: {modelo_nombre}  |  Clases: OLIMPICO / PARALIMPICO",
                         font=('Segoe UI', 9, 'italic'), background='#F5F5F5',
                         foreground='#666')
        subt.pack(pady=(8, 16))

        # Frame de inputs
        frame = ttk.Frame(root, padding=20)
        frame.pack(fill='x', padx=20)

        # Vigencia
        ttk.Label(frame, text="Vigencia (anio):").grid(row=0, column=0, sticky='w', pady=8)
        self.vig_var = tk.StringVar(value=str(VIGENCIAS[-1]))
        ttk.Combobox(frame, textvariable=self.vig_var,
                     values=[str(v) for v in VIGENCIAS],
                     state='readonly', width=25).grid(row=0, column=1, pady=8)

        # Grupo etario
        ttk.Label(frame, text="Grupo etario:").grid(row=1, column=0, sticky='w', pady=8)
        self.gru_var = tk.StringVar(value=GRUPOS_ETARIOS[2])
        ttk.Combobox(frame, textvariable=self.gru_var, values=GRUPOS_ETARIOS,
                     state='readonly', width=25).grid(row=1, column=1, pady=8)

        # Inversion
        ttk.Label(frame, text="Inversion por deportista (COP):").grid(row=2, column=0,
                                                                       sticky='w', pady=8)
        self.inv_var = tk.StringVar(value="3310519.10")
        ttk.Entry(frame, textvariable=self.inv_var, width=27).grid(row=2, column=1, pady=8)

        # Botones
        btn_frame = ttk.Frame(root, padding=10)
        btn_frame.pack(pady=10)
        ttk.Button(btn_frame, text="Predecir",
                   command=self.predecir).pack(side='left', padx=8, ipadx=14, ipady=4)
        ttk.Button(btn_frame, text="Limpiar",
                   command=self.limpiar).pack(side='left', padx=8, ipadx=14, ipady=4)

        # Resultado
        self.resultado_lbl = ttk.Label(root, text="", font=('Segoe UI', 13, 'bold'),
                                        background='#F5F5F5')
        self.resultado_lbl.pack(pady=12)
        self.proba_lbl = ttk.Label(root, text="", font=('Segoe UI', 10),
                                    background='#F5F5F5', foreground='#444')
        self.proba_lbl.pack()

        # Footer
        ttk.Label(root, text="UPB - Analitica de Datos - Practica 4",
                  font=('Segoe UI', 8), background='#F5F5F5',
                  foreground='#888').pack(side='bottom', pady=8)

    def predecir(self):
        try:
            vig = int(self.vig_var.get())
            gru = self.gru_var.get()
            inv = float(self.inv_var.get())
            if inv < 0:
                raise ValueError("La inversion no puede ser negativa")

            clase, proba = predecir(vig, gru, inv)
            color = '#2E75B6' if clase == 'OLIMPICO' else '#E97132'
            self.resultado_lbl.config(text=f"Prediccion: {clase}", foreground=color)
            self.proba_lbl.config(
                text=f"P(PARALIMPICO) = {proba:.4f}   |   P(OLIMPICO) = {1-proba:.4f}")
        except ValueError as e:
            messagebox.showerror("Error de entrada", f"Verifica los datos:\n{e}")
        except Exception as e:
            messagebox.showerror("Error", str(e))

    def limpiar(self):
        self.vig_var.set(str(VIGENCIAS[-1]))
        self.gru_var.set(GRUPOS_ETARIOS[2])
        self.inv_var.set("3310519.10")
        self.resultado_lbl.config(text="")
        self.proba_lbl.config(text="")


# Para lanzar la GUI desde Jupyter:
# root = tk.Tk()
# app = PredictorApp(root)
# root.mainloop()

print("Clase PredictorApp definida. Para abrir la ventana, descomenta las 3 lineas finales.")

Clase PredictorApp definida. Para abrir la ventana, descomenta las 3 lineas finales.


## 4. Exportar la app como script standalone

Genera un archivo `app_tkinter.py` que se puede ejecutar fuera de Jupyter con `python app_tkinter.py`. Útil para:
- Tomar la captura de pantalla del despliegue.
- Distribuir la aplicación sin Jupyter.

In [4]:
script = '''"""
App Tkinter - Predictor Sector Escuela de Talentos
UPB - Analitica de Datos - Practica 4
Ejecutar:  python app_tkinter.py
"""
import joblib
import pandas as pd
import tkinter as tk
from tkinter import ttk, messagebox

# Cargar artefactos
artefactos = joblib.load("modelo_final.joblib")
modelo            = artefactos["modelo"]
scaler            = artefactos["scaler"]
label_encoder     = artefactos["label_encoder"]
feature_columns   = artefactos["feature_columns"]
requires_scaling  = artefactos["requires_scaling"]
modelo_nombre     = artefactos["modelo_nombre"]

GRUPOS_ETARIOS = [
    "Primera Infancia 0-5", "Infancia 6-11", "Adolescencia 12-18",
    "Juventud 19-26", "Adultez 27-59",
]
VIGENCIAS = [2022, 2023, 2024]


def predecir(vigencia, grupo_etario, inversion):
    fila = pd.DataFrame([{c: 0 for c in feature_columns}])
    fila["VIGENCIA"] = vigencia
    fila["Inversion_por_deportista"] = inversion
    col = f"Grupo_Etario_{grupo_etario.replace(' ', '_').replace('-', '_')}"
    if col not in fila.columns:
        raise ValueError(f"Grupo etario invalido: {grupo_etario}")
    fila[col] = 1
    fila = fila[feature_columns]
    X = scaler.transform(fila) if requires_scaling else fila
    idx = modelo.predict(X)[0]
    clase = label_encoder.inverse_transform([idx])[0]
    proba = modelo.predict_proba(X)[0][int(label_encoder.transform(["PARALIMPICO"])[0])]
    return clase, float(proba)


class PredictorApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Predictor Sector - Escuela de Talentos")
        self.root.geometry("520x520")
        self.root.configure(bg="#F5F5F5")

        style = ttk.Style(); style.theme_use("clam")
        style.configure("TLabel", background="#F5F5F5", font=("Segoe UI", 10))
        style.configure("TButton", font=("Segoe UI", 10, "bold"))
        style.configure("Header.TLabel", font=("Segoe UI", 14, "bold"),
                       background="#2E75B6", foreground="white")

        ttk.Label(root, text="  Predictor de Sector Deportivo",
                  style="Header.TLabel", anchor="w").pack(fill="x", ipady=12)
        ttk.Label(root,
                  text=f"Modelo: {modelo_nombre}  |  Clases: OLIMPICO / PARALIMPICO",
                  font=("Segoe UI", 9, "italic"), background="#F5F5F5",
                  foreground="#666").pack(pady=(8, 16))

        frame = ttk.Frame(root, padding=20); frame.pack(fill="x", padx=20)

        ttk.Label(frame, text="Vigencia (anio):").grid(row=0, column=0, sticky="w", pady=8)
        self.vig_var = tk.StringVar(value=str(VIGENCIAS[-1]))
        ttk.Combobox(frame, textvariable=self.vig_var, values=[str(v) for v in VIGENCIAS],
                     state="readonly", width=25).grid(row=0, column=1, pady=8)

        ttk.Label(frame, text="Grupo etario:").grid(row=1, column=0, sticky="w", pady=8)
        self.gru_var = tk.StringVar(value=GRUPOS_ETARIOS[2])
        ttk.Combobox(frame, textvariable=self.gru_var, values=GRUPOS_ETARIOS,
                     state="readonly", width=25).grid(row=1, column=1, pady=8)

        ttk.Label(frame, text="Inversion por deportista (COP):").grid(row=2, column=0,
                                                                       sticky="w", pady=8)
        self.inv_var = tk.StringVar(value="3310519.10")
        ttk.Entry(frame, textvariable=self.inv_var, width=27).grid(row=2, column=1, pady=8)

        btn = ttk.Frame(root, padding=10); btn.pack(pady=10)
        ttk.Button(btn, text="Predecir",
                   command=self.predecir).pack(side="left", padx=8, ipadx=14, ipady=4)
        ttk.Button(btn, text="Limpiar",
                   command=self.limpiar).pack(side="left", padx=8, ipadx=14, ipady=4)

        self.res = ttk.Label(root, text="", font=("Segoe UI", 13, "bold"),
                             background="#F5F5F5"); self.res.pack(pady=12)
        self.pro = ttk.Label(root, text="", font=("Segoe UI", 10),
                             background="#F5F5F5", foreground="#444"); self.pro.pack()
        ttk.Label(root, text="UPB - Analitica de Datos - Practica 4",
                  font=("Segoe UI", 8), background="#F5F5F5",
                  foreground="#888").pack(side="bottom", pady=8)

    def predecir(self):
        try:
            vig = int(self.vig_var.get())
            gru = self.gru_var.get()
            inv = float(self.inv_var.get())
            if inv < 0:
                raise ValueError("La inversion no puede ser negativa")
            clase, p = predecir(vig, gru, inv)
            color = "#2E75B6" if clase == "OLIMPICO" else "#E97132"
            self.res.config(text=f"Prediccion: {clase}", foreground=color)
            self.pro.config(
                text=f"P(PARALIMPICO) = {p:.4f}   |   P(OLIMPICO) = {1-p:.4f}")
        except ValueError as e:
            messagebox.showerror("Error de entrada", f"Verifica los datos:\\n{e}")
        except Exception as e:
            messagebox.showerror("Error", str(e))

    def limpiar(self):
        self.vig_var.set(str(VIGENCIAS[-1]))
        self.gru_var.set(GRUPOS_ETARIOS[2])
        self.inv_var.set("3310519.10")
        self.res.config(text=""); self.pro.config(text="")


if __name__ == "__main__":
    root = tk.Tk()
    PredictorApp(root)
    root.mainloop()
'''

with open('app_tkinter.py', 'w', encoding='utf-8') as f:
    f.write(script)

print("Script standalone exportado: app_tkinter.py")
print("Para ejecutar:  python app_tkinter.py")

Script standalone exportado: app_tkinter.py
Para ejecutar:  python app_tkinter.py


## 5. Pruebas de la función de predicción (sin GUI)

Validamos que el pipeline de predicción funciona con casos representativos antes del despliegue gráfico.

In [5]:
# Casos de prueba
casos = [
    (2024, 'Adolescencia 12-18', 3310519.10),
    (2024, 'Infancia 6-11',      3310519.10),
    (2024, 'Adolescencia 12-18', 16267579.69),
    (2023, 'Juventud 19-26',     5160151.32),
    (2022, 'Primera Infancia 0-5', 1872659.18),
]

print(f"{'Vigencia':<10}{'Grupo Etario':<25}{'Inversion':>15}  -> {'Prediccion':<15}{'P(PARA)':>10}")
print("-" * 90)
for vig, gru, inv in casos:
    clase, p = predecir(vig, gru, inv)
    print(f"{vig:<10}{gru:<25}{inv:>15,.0f}  -> {clase:<15}{p:>10.4f}")

Vigencia  Grupo Etario                   Inversion  -> Prediccion        P(PARA)
------------------------------------------------------------------------------------------
2024      Adolescencia 12-18             3,310,519  -> OLIMPICO           0.4694
2024      Infancia 6-11                  3,310,519  -> PARALIMPICO        1.0000
2024      Adolescencia 12-18            16,267,580  -> PARALIMPICO        1.0000
2023      Juventud 19-26                 5,160,151  -> PARALIMPICO        1.0000
2022      Primera Infancia 0-5           1,872,659  -> PARALIMPICO        1.0000


## 6. Probar el despliegue

Para probar el despliegue:

1. Abre una terminal en la carpeta del proyecto.
2. Asegúrate de tener `modelo_final.joblib` (generado por el Notebook 2).
3. Ejecuta:
   ```bash
   python app_tkinter.py
   ```
4. Llena los campos, presiona **Predecir**.


---

## 7. Resumen del despliegue

- **Modelo en producción:** {ver `best_params` cargado al inicio del notebook}
- **Tecnología de UI:** Tkinter (estándar de Python — sin dependencias externas).
- **Inputs del usuario:** Vigencia, Grupo etario, Inversión por deportista.
- **Outputs:** Sector predicho + probabilidad PARALIMPICO.
- **Validaciones de entrada:** valores negativos rechazados, dropdowns para evitar valores inválidos en los categóricos.
